<style>
.jp-RenderedHTMLCommon h1 { color:#8b1e3f; font-size:2.35em; }
.jp-RenderedHTMLCommon h2 { color:#345995; }
.jp-RenderedHTMLCommon h3 { color:#374151; }
.jp-RenderedHTMLCommon blockquote { border-left:6px solid #8b1e3f; background:#fff7f8; padding:.65em 1em; }
.jp-RenderedHTMLCommon table { font-size:.91em; }
.jp-RenderedHTMLCommon code { color:#9a3412; }
.jp-RenderedHTMLCommon pre { border:1px solid #94a3b8; background:#f8fafc; padding:1em; }
</style>

# Amazon Redshift: Introduction
## Data warehousing, deployment models, and internal architecture

**Provisioned clusters · Serverless · MPP · Columnar execution · Managed storage**

# Learning outcomes

After this lesson, you can:

- explain why Redshift is a data warehouse rather than an OLTP database;
- compare provisioned and serverless operating models;
- trace a query through leader/coordinator, compute, slices, and storage;
- connect MPP, columnar storage, compression, distribution, and sort order;
- identify the responsibilities AWS manages and those the customer retains;
- recognize which ideas in older Redshift material remain valid and which are obsolete.

# What is a data warehouse?

A data warehouse organizes integrated, historical data for analytics. It is designed for questions such as:

- How did revenue change across regions and quarters?
- Which customer segments have the highest lifetime value?
- Which operational patterns predict failure or churn?

Typical warehouse work scans many rows, joins large relations, and performs aggregates. That workload differs fundamentally from recording one order or updating one account balance.

# OLTP and analytical workloads

| Characteristic | OLTP system | Analytical warehouse |
|---|---|---|
| Main purpose | run business transactions | analyze the business |
| Typical request | point lookup or small update | scan, join, aggregate |
| Rows touched | few | thousands to billions |
| Data shape | normalized operational model | dimensional, wide, or curated model |
| Concurrency | many short transactions | mixed BI, ELT, reporting, data science |
| Optimization focus | latency and consistency per transaction | throughput and time-to-insight |

> Redshift supports SQL transactions, but it is optimized for analytical processing—not as a replacement for a high-frequency application database.

# Amazon Redshift in one sentence

Amazon Redshift is a managed, relational, SQL data-warehouse service designed for large-scale analytics using massively parallel processing and column-oriented execution.

It can work with:

- data stored in Redshift-managed tables;
- data in Amazon S3 and other supported sources;
- ingestion and transformation pipelines;
- JDBC/ODBC clients, BI tools, notebooks, and applications.

> PostgreSQL heritage improves SQL and driver familiarity, but Redshift is not PostgreSQL and should not be evaluated as though the engines were interchangeable.

# Two deployment models

```text
+---------------------------+       +---------------------------+
| Provisioned               |       | Serverless                |
|                           |       |                           |
| Choose cluster/node       |       | Choose workload controls |
| capacity and resize it    |       | and RPU capacity settings |
|                           |       |                           |
| Stable, explicit compute  |       | Compute scales for demand |
+-------------+-------------+       +-------------+-------------+
              |                                   |
              +--------------+--------------------+
                             v
                 Redshift SQL analytics platform
```

Both models expose Redshift SQL capabilities. The major difference is how compute capacity and its operational boundary are expressed.

# Provisioned Redshift

A provisioned data warehouse is a **cluster**: a selected node type and node count running the Redshift engine.

You decide or govern:

- cluster size and node family;
- network placement and endpoint access;
- resize strategy and maintenance preferences;
- workload management, monitoring, and cost commitments.

AWS manages underlying infrastructure operations, patches, failure handling, backups, and service integration. You still own schema, data, permissions, workload design, and cost controls.

# Provisioned cluster architecture

```text
+----------------------+
| BI / SQL / ETL client|
+----------+-----------+
           | SQL over endpoint
           v
+----------------------+
| Leader node          |  parse · optimize · coordinate · return results
+----------+-----------+
           | parallel plan segments
     +-----+----------------------+
     v                            v
+----+-------------+        +-----+------------+
| Compute node 1   | <----> | Compute node N  |
| slices work in   | network| slices work in  |
| parallel         | exchange| parallel        |
+------------------+        +------------------+
```

Clients communicate with the endpoint; compute nodes are not addressed individually.

# Leader node responsibilities

For queries involving compute-node tables, the leader node typically:

1. accepts and parses SQL;
2. resolves objects and validates the request;
3. creates an MPP-aware execution plan;
4. compiles and distributes plan segments;
5. coordinates exchanges between compute nodes;
6. aggregates final intermediate results;
7. returns the result to the client.

> The leader is a coordinator, not the place where every byte of every table is serially processed.

# Compute nodes and slices

A compute node supplies CPU and memory. Redshift partitions its work into **slices**. Each slice receives a share of node resources and a portion of the data/workload.

```text
+---------------- Compute node ----------------+
| +------------+ +------------+ +------------+ |
| | Slice 0    | | Slice 1    | | Slice ...  | |
| | data part A| | data part B| | data part N| |
| | plan work A| | plan work B| | plan work N| |
| +------------+ +------------+ +------------+ |
+----------------------------------------------+
```

Balanced work across slices is crucial. If one slice receives far more data or work, the query completes at the speed of that straggler.

# Massively parallel processing (MPP)

MPP divides a large operation into plan segments that execute concurrently over data partitions.

```text
Large scan + join + aggregate
              |
      split into parallel work
       /         |          \
  Slice A     Slice B      Slice C
       \         |          /
        exchange / combine
              |
         final result
```

Parallelism helps when data and work are well distributed. More capacity cannot fully compensate for severe skew, unnecessary scans, or expensive redistribution.

# Why columnar storage matters

Analytical queries often need a few columns from many rows. Column-oriented storage groups values by column so the engine can avoid reading unrelated attributes.

```text
Row-oriented block:    [id, date, region, product, amount] x many rows

Column-oriented:       [amount, amount, amount, ...]
                       [region, region, region, ...]
```

Benefits include:

- less I/O through column pruning;
- effective compression of similar adjacent values;
- vector-friendly scans and aggregates.

# Compression and data blocks

Redshift stores column values in blocks and applies column-specific encodings. Compression is not merely a storage-saving feature: fewer physical bytes generally means less I/O.

A good design asks:

- Does the encoding suit the values and cardinality?
- Are loads allowing Redshift to choose or preserve useful encodings?
- Are statistics representative enough for the optimizer?

> Prefer supported automatic optimization features and evidence from the workload over blindly applying encoding advice from old node generations.

# Data distribution

Distribution determines where table rows live across slices. It affects both balance and the amount of data moved during joins and aggregates.

| Strategy idea | Intent | Risk |
|---|---|---|
| even distribution | spread rows across slices | related rows may move for joins |
| key distribution | colocate rows sharing a key | skew if the key is uneven |
| replicate a small table | make it locally available | storage/load overhead if not small |
| automatic choice | let Redshift adapt the choice | still monitor observed plans and skew |

The right choice is workload-dependent, not a universal rule.

# Join locality

```text
Colocated join                         Redistributed join

Slice 1: orders A + items A            Slice 1: orders A + items C
Slice 2: orders B + items B            Slice 2: orders B + items A
             |                                      |
        local matching                       network exchange
                                                    |
                                              matching can begin
```

Network redistribution is sometimes necessary. The goal is not to eliminate every exchange, but to avoid repeated large exchanges on dominant workloads.

# Sort order and block elimination

Sort keys influence how rows are ordered in storage. Metadata such as zone maps records value ranges for blocks, allowing Redshift to skip blocks that cannot satisfy a predicate.

```sql
SELECT region, SUM(net_amount)
FROM fact_sales
WHERE sale_date >= DATE '2026-08-01'
GROUP BY region;
```

If storage organization aligns with common filters, older blocks may be skipped. Sort design should follow real predicates and joins, and automatic table optimization may manage choices for eligible tables.

# Redshift Managed Storage

Modern managed-storage node families separate the compute decision from the full logical data size. Frequently accessed blocks can use local SSD caching while durable managed storage scales beyond that cache.

```text
+-------------------------------+
| Compute nodes                 |
| CPU + memory + local SSD cache|
+---------------+---------------+
                | automatic block placement
                v
+-------------------------------+
| Redshift Managed Storage      |
| durable storage backed by S3  |
+-------------------------------+
```

This is materially different from the old deck's storage-bound DS node model.

# Redshift Serverless

Redshift Serverless removes cluster and node selection from the customer's normal operating model. Compute capacity scales to serve workloads and is measured in **Redshift Processing Units (RPUs)**.

You still configure and govern:

- database objects and data;
- namespace and workgroup settings;
- VPC access, security groups, IAM, and encryption;
- base/max capacity and usage limits;
- query design, workload isolation, monitoring, and cost.

> Serverless removes node administration; it does not remove capacity, networking, security, or cost decisions.

# Serverless: namespace and workgroup

```text
+-----------------------------+     one-to-one     +-----------------------------+
| Namespace                   | <----------------> | Workgroup                   |
|                             |                    |                             |
| databases, schemas, tables  |                    | RPUs / compute controls     |
| users, KMS key, recovery    |                    | endpoint, VPC, security     |
| data shares, usage settings |                    | limits and configuration    |
+-----------------------------+                    +-----------------------------+
       storage/logical plane                              compute/access plane
```

Separating these concepts makes it easier to reason about persistent database state versus the compute boundary that serves queries.

# Serverless query path

```text
+-------------+      +-------------------+      +--------------------+
| SQL client  | ---> | Workgroup endpoint| ---> | Managed Redshift   |
+-------------+      | network + compute |      | parallel execution |
                     +-------------------+      +---------+----------+
                                                               |
                                                               v
                                                    +--------------------+
                                                    | Namespace data and |
                                                    | database objects   |
                                                    +--------------------+
```

The service still performs distributed analytical execution. Serverless changes the operational abstraction presented to you; it does not turn warehouse queries into single-process execution.

# Provisioned or serverless?

| Decision dimension | Provisioned | Serverless |
|---|---|---|
| Capacity expression | node family and count | RPU-based workgroup capacity |
| Scaling control | explicit resize and related features | service scaling within configured controls |
| Operational model | stable cluster boundary | managed compute boundary |
| Cost reasoning | provisioned capacity over time | consumed capacity and configured limits |
| Good starting signal | predictable, steady, tightly managed workloads | variable, intermittent, or rapidly changing demand |

This is not a permanent one-way choice. Evaluate workload shape, isolation, governance, feature/Region requirements, and measured cost—not only ease of initial setup.

# Loading data

Warehouses are most efficient when data arrives in parallel, set-oriented operations rather than long streams of tiny row inserts.

```text
Operational sources / streams / files
                  |
          land or stage data
                  v
        Amazon S3 / integrations
                  |
        parallel COPY / ingestion
                  v
       Redshift staging and tables
                  |
        SQL transformations / MERGE
                  v
          curated warehouse model
```

# Workload management

A warehouse normally serves competing work:

- dashboard queries requiring predictable response;
- ad hoc analyst exploration;
- scheduled ELT and maintenance;
- data-science feature preparation;
- administrative and monitoring queries.

Workload management establishes queues, priorities, isolation, and resource behavior. Capacity planning without workload governance can allow one expensive query class to degrade every other consumer.

# Security boundaries

Ask these questions separately:

1. **Who may call the service?** IAM and control-plane permissions.
2. **Who may connect to SQL?** Database identity, credentials, federation, and endpoint reachability.
3. **What may they query?** Database grants, roles, row/column controls, and shared-data permissions.
4. **How does traffic flow?** VPC subnets, routing, security groups, and enhanced VPC routing where applicable.
5. **How is data protected?** Encryption in transit and at rest, KMS keys, logging, and recovery controls.

> A reachable endpoint is not authorization, and a database grant is not network reachability.

# Reliability and recovery

Managed service does not mean data management is automatic in every business sense. Plan for:

- automated and manual recovery points or snapshots;
- retention that matches recovery objectives;
- cross-Region or cross-account strategy where required;
- tested restoration, not merely configured backup;
- protection against accidental logical changes;
- upstream replayability and downstream reconciliation.

Infrastructure failure recovery and recovery from a bad `DELETE`, faulty pipeline, or incorrect business transformation are different problems.

# Performance is a system property

Query time can be dominated by different constraints:

| Symptom | Possible dominant cause |
|---|---|
| large scan | weak pruning, excessive columns, poor storage layout |
| long network phase | redistribution, large intermediates, skew |
| one slow slice | uneven data or computation |
| queue delay | workload contention or capacity policy |
| bad join order | stale/missing statistics or difficult predicates |
| repeated expensive work | missing materialization, cache opportunity, or aggregate design |

Start with the execution evidence; do not begin by randomly resizing or adding keys.

# What the historical AWS deck gets right

The attached older deck remains useful for these durable concepts:

- Redshift is optimized for data warehousing;
- parallelism and distribution drive query behavior;
- leader/coordinator and compute responsibilities differ;
- columnar storage, compression, and block elimination reduce I/O;
- bulk parallel loading is preferable to row-at-a-time ingestion;
- distribution and sort design affect joins and scans;
- backup, scaling, monitoring, and security are part of warehouse operations.

These ideas were used as conceptual prompts, then checked against current AWS documentation.

# What not to copy from the old deck

Treat the following as historical, not current guidance:

- DC1, DS1, and DS2 sizing or capacity tables;
- decade-old price claims and free-trial statements;
- old console screenshots and provisioning workflow;
- claims that Redshift lacks a single-statement merge capability;
- Python 2.7 UDF positioning;
- old assumptions that storage must scale with node-attached disks;
- advice that ignores automatic table optimization and current node families;
- product availability statements from the deck's publication era.

> Architecture concepts age more slowly than instance names, limits, prices, interfaces, and feature lists.

# End-to-end mental model

```text
Sources -> ingestion -> warehouse tables / lake data -> SQL endpoint -> consumers
                 |                |                     |
                 |                |                     +-> BI / notebooks / apps
                 |                +-> distribution · sort · compression · statistics
                 +-> parallel, set-oriented loading

Across the whole path:
security · networking · workload policy · monitoring · recovery · cost governance
```

Provisioned and serverless change the compute-management boundary, but both require sound data and workload engineering.

# Knowledge check

1. Why does columnar storage suit warehouse queries?
2. What does the leader node coordinate, and what do compute slices execute?
3. How can data skew reduce the benefit of MPP?
4. Why can a poor distribution choice make a join network-bound?
5. What is the difference between a Serverless namespace and workgroup?
6. Which customer decisions remain in a serverless deployment?
7. When might a provisioned cluster be preferable?
8. Why is the attached deck unsafe as a source for current node types or prices?

# Recap

**Amazon Redshift combines:**

1. relational SQL for analytical consumers;
2. MPP execution across compute resources and slices;
3. columnar storage, compression, and block pruning;
4. data placement choices that influence locality and balance;
5. managed storage that can scale separately from compute;
6. provisioned and serverless capacity models;
7. managed infrastructure with customer-owned data, security, workload, and cost design.

> The shortest useful model: **Redshift is a managed, distributed SQL analytics system—not simply PostgreSQL hosted on a large server.**

# References

Current AWS documentation:

- [Amazon Redshift architecture](https://docs.aws.amazon.com/redshift/latest/dg/c_redshift_system_overview.html)
- [Data warehouse system architecture](https://docs.aws.amazon.com/redshift/latest/dg/c_high_level_system_architecture.html)
- [Provisioned clusters](https://docs.aws.amazon.com/redshift/latest/mgmt/working-with-clusters.html)
- [Serverless workgroups and namespaces](https://docs.aws.amazon.com/redshift/latest/mgmt/serverless-workgroup-namespace.html)
- [Serverless compute capacity](https://docs.aws.amazon.com/redshift/latest/mgmt/serverless-capacity.html)
- [Amazon Redshift performance](https://docs.aws.amazon.com/redshift/latest/dg/c_challenges_achieving_high_performance_queries.html)

Historical reference supplied by the learner:

- `AWS_14_Redshift.pdf` — AWS-branded masterclass material from the earlier Redshift era; used for durable concepts only.